# LAB 03 - Word Representations and Embeddings

Trong notebook này, em khảo sát cách dùng ngữ cảnh để biểu diễn từ. Em
bắt đầu từ ma trận co-occurrence, sau đó dùng Word2Vec để thử similarity,
analogy và tìm kiếm văn bản.

Phần tính toán nằm trong `calculations.md`, dự đoán ban đầu trong
`prediction.md`. Em dùng môi trường `NLP` ở root với kernel **Python (NLP)**.
Bảng tổng hợp trong README lấy từ `results.csv` của lượt chạy mới nhất;
thời gian train có thể khác output đã lưu ở đây. Thông tin hỗ trợ công
cụ được ghi ở cuối README.

In [1]:
from pathlib import Path
import json, inspect, sys
from experiments import *
recorder = Recorder()
environment = save_environment()
print(json.dumps(environment, indent=2))

{
  "python": "3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)]",
  "platform": "Windows-11-10.0.26200-SP0",
  "packages": {
    "numpy": "2.5.3",
    "scipy": "1.18.1",
    "gensim": "4.4.0",
    "sklearn": "1.9.1",
    "nbformat": "5.11.1",
    "nbclient": "0.11.0",
    "ipykernel": "7.4.0"
  },
  "pythonhashseed": "42"
}


## 1. Biểu diễn word-context

Em đếm context ở hai phía của target và không vượt biên câu. Window là
bán kính tính theo vị trí token. Em giữ stopwords trong phần này và dùng
cùng một vocabulary khi thử window 1, 2, 5.

Các hàm vocabulary, co-occurrence, cosine và most_similar được viết trong
`cooccurrence.py`. Em thử thêm PPMI + SVD 3 chiều để minh họa cách nén
ma trận đếm thành biểu diễn dense.

In [2]:
from cooccurrence import build_vocabulary, build_cooccurrence_matrix, cosine_similarity, most_similar
print(inspect.getsource(build_cooccurrence_matrix))
cooccurrence_results = cooccurrence_experiment(recorder)

def build_cooccurrence_matrix(sentences, vocabulary, window=1):
    """Đếm context hai phía, trọng số 1; không vượt biên câu.

    Từ ngoài vocabulary vẫn chiếm một vị trí trong câu, nhưng không được
    đếm vào ma trận. Không xóa chúng trước khi xác định khoảng cách.
    """
    if not isinstance(window, Integral) or window < 1:
        raise ValueError("window must be a positive integer")
    if set(vocabulary.values()) != set(range(len(vocabulary))):
        raise ValueError("vocabulary indices must be contiguous from zero")
    matrix = np.zeros((len(vocabulary), len(vocabulary)), dtype=np.int64)
    for sentence in sentences:
        for i, word in enumerate(sentence):
            if word not in vocabulary:
                continue
            for j in range(max(0, i-window), min(len(sentence), i+window+1)):
                if j != i and sentence[j] in vocabulary:
                    matrix[vocabulary[word], vocabulary[sentence[j]]] += 1
    return matrix

{'window': 1, 'vocabular

{'window': 2, 'vocabulary': 26, 'shape': [26, 26], 'nonzero': 114, 'entries': 676, 'cat_dog': 0.8749999999999998, 'doctor_physician': 1.0, 'doctor_hospital': 0.819891591749923, 'doctor_top5': [('physician', 1.0), ('works', 0.8616404368553291), ('hospital', 0.819891591749923), ('patient', 0.819891591749923), ('treats', 0.7385489458759964)]}


{'window': 5, 'vocabulary': 26, 'shape': [26, 26], 'nonzero': 149, 'entries': 676, 'cat_dog': 0.8749999999999998, 'doctor_physician': 1.0, 'doctor_hospital': 0.8495907904857465, 'doctor_top5': [('physician', 1.0), ('in', 0.8559209850218259), ('works', 0.8559209850218259), ('hospital', 0.8495907904857465), ('treats', 0.8374357893586237)]}


In [3]:
toy = [tokenize(s) for s in TOY_TEXTS]
vocab = build_vocabulary(toy)
X = build_cooccurrence_matrix(toy, vocab, window=2)
print('doctor top-5:', most_similar('doctor', X, vocab, top_k=5))
print('Vocabulary:', vocab)
print('Matrix:')
print(X)

doctor top-5: [('physician', 1.0), ('works', 0.8616404368553291), ('hospital', 0.819891591749923), ('patient', 0.819891591749923), ('treats', 0.7385489458759964)]
Vocabulary: {'a': 0, 'banana': 1, 'cat': 2, 'computer': 3, 'doctor': 4, 'dog': 5, 'eats': 6, 'fish': 7, 'fruit': 8, 'hospital': 9, 'in': 10, 'is': 11, 'likes': 12, 'meat': 13, 'medical': 14, 'milk': 15, 'needs': 16, 'patient': 17, 'physician': 18, 'program': 19, 'provides': 20, 'runs': 21, 'the': 22, 'treatment': 23, 'treats': 24, 'works': 25}
Matrix:
[[0 1 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 2 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 3 0 1 1]
 [0 0 0 0 0 0 1 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 2 0 0 0]
 [0 0 1 0 0 1 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 2 0 0 0]
 [0 0 1 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0

## 2. Corpus và tiền xử lý

Em lấy 6.000 document đầu tiên trong file
`datasets/c4-train.00000-of-01024-30K.json`, theo thứ tự file, không lọc
riêng theo từ cần đánh giá.

Các bước gồm chuyển chữ thường, tách câu tại `. ! ?` và xuống dòng,
lấy token bằng `[a-z]+`, giữ câu có ít nhất 3 token và giữ stopwords.
Sau xử lý có 128.104 câu, 2.224.629 token; với min_count=3, Word2Vec giữ
29.367 từ.

Cách tách này có thể chưa xử lý tốt viết tắt, số và từ ghép. Doc_id và
URL giúp em xem lại câu nguồn, nhưng không được đưa vào model như token.

In [4]:
sentences, records, metadata, counts = prepare_corpus()
print('Total training tokens:', sum(map(len, sentences)))

{
  "source_file": "c4-train.00000-of-01024-30K.json",
  "source_sha256": "568da6a0ba4750a9e20a55a06dfba1c18dd057dc21f3e20c240183abfd912988",
  "selection": "first N documents in original file order; no keyword filtering",
  "requested_documents": 6000,
  "source_documents": 6000,
  "sentences": 128104,
  "tokens": 2224629,
  "raw_vocabulary": 67927,
  "min_count": 3,
  "retained_vocabulary": 29367,
  "preprocessing": "lowercase; split on . ! ? newline; [a-z]+ tokens; >=3 tokens/sentence; keep stopwords",
  "corpus_sha256": "fe28252c899d883b2e90a879e8d9625295b167b551db576307da2ba4cc568e4e"
}


Probe frequencies: {'automobile': 12, 'banana': 31, 'bank': 250, 'car': 671, 'cat': 107, 'computer': 341, 'disease': 233, 'doctor': 204, 'dog': 305, 'football': 162, 'hospital': 234, 'king': 214, 'patient': 232, 'physician': 65, 'queen': 105}


Total training tokens: 2224629


## 3. Huấn luyện Word2Vec

Em dùng Skip-gram với negative sampling và thử 5 cấu hình:
`(dimension, window) = (100,5), (100,2), (100,10), (50,5), (300,5)`.
Model 100 chiều, window=5 là baseline và được dùng chung cho hai phép
so sánh window và dimension.

Các tham số còn lại: `sg=1`, `negative=5`, `hs=0`, `min_count=3`,
`epochs=8`, `sample=0.001`, `workers=1`, `seed=42`,
`shrink_windows=False`, `sorted_vocab=1`. Learning rate giảm từ 0,025
đến 0,0001. Em không dùng pretrained vectors.

Thời gian đo gồm xây vocabulary và train. Kích thước model là dung
lượng file lưu; kích thước vectors chỉ tính mảng input embeddings.
Em mới thử một seed nên chưa đánh giá được độ dao động giữa các lượt.

In [5]:
print(inspect.getsource(train_experiments))
models, summaries = train_experiments(sentences, recorder)

def train_experiments(sentences, recorder):
    configurations = [(100, 5), (100, 2), (100, 10), (50, 5), (300, 5)]
    summaries, models = {}, {}
    (ROOT / 'models').mkdir(exist_ok=True)
    for dimension, window in configurations:
        model_id = f'sg_d{dimension}_w{window}'
        print('TRAIN', model_id, flush=True)
        start = time.perf_counter()
        model = Word2Vec(sentences=sentences, vector_size=dimension, window=window,
                         min_count=MIN_COUNT, epochs=EPOCHS, workers=1, sg=1,
                         negative=5, hs=0, sample=1e-3, seed=SEED,
                         shrink_windows=False, sorted_vocab=1)
        training_seconds = time.perf_counter() - start
        model_path = ROOT / 'models' / f'{model_id}.model'
        model.save(str(model_path), separately=[])
        metrics = evaluate_model(model, recorder, model_id)
        metrics.update(dimension=dimension, window=window, epochs=EPOCHS,
                       min_count=MIN_COUNT, v

DONE sg_d100_w5 seconds= 61.4 analogy= {'covered': 8, 'total': 8, 'accuracy_at_1': 0.25, 'accuracy_at_5': 0.375} search= {'tfidf': {'mrr': 0.8333333333333334, 'recall_at_3': 0.6388888888888888}, 'embedding': {'mrr': 1.0, 'recall_at_3': 1.0}}


TRAIN sg_d100_w2


DONE sg_d100_w2 seconds= 25.54 analogy= {'covered': 8, 'total': 8, 'accuracy_at_1': 0.125, 'accuracy_at_5': 0.5} search= {'tfidf': {'mrr': 0.8333333333333334, 'recall_at_3': 0.6388888888888888}, 'embedding': {'mrr': 1.0, 'recall_at_3': 1.0}}


TRAIN sg_d100_w10


DONE sg_d100_w10 seconds= 81.59 analogy= {'covered': 8, 'total': 8, 'accuracy_at_1': 0.0, 'accuracy_at_5': 0.375} search= {'tfidf': {'mrr': 0.8333333333333334, 'recall_at_3': 0.6388888888888888}, 'embedding': {'mrr': 1.0, 'recall_at_3': 1.0}}


TRAIN sg_d50_w5


DONE sg_d50_w5 seconds= 49.14 analogy= {'covered': 8, 'total': 8, 'accuracy_at_1': 0.125, 'accuracy_at_5': 0.25} search= {'tfidf': {'mrr': 0.8333333333333334, 'recall_at_3': 0.6388888888888888}, 'embedding': {'mrr': 1.0, 'recall_at_3': 1.0}}


TRAIN sg_d300_w5


DONE sg_d300_w5 seconds= 95.42 analogy= {'covered': 8, 'total': 8, 'accuracy_at_1': 0.125, 'accuracy_at_5': 0.5} search= {'tfidf': {'mrr': 0.8333333333333334, 'recall_at_3': 0.6388888888888888}, 'embedding': {'mrr': 1.0, 'recall_at_3': 1.0}}


## 4. Các từ gần nhất và context nguồn

Em lấy Top-5 cho doctor, hospital, patient, disease, computer, football
và banana, rồi xem thêm tần suất và câu nguồn.

Ở baseline, physician đứng đầu danh sách của doctor. Các từ dentist,
surgeon và ophthalmologist cũng có liên quan đến nghề khám chữa bệnh.
Anesthesia đứng thứ hai nhưng chỉ việc gây mê, nên em xem danh sách này
là sự kết hợp của gần nghĩa và cùng chủ đề.

Các context chung doctor/physician có care, medical, consult và medicine.
Em dùng chúng để hỗ trợ diễn giải, chưa coi đây là bằng chứng về nguyên
nhân duy nhất tạo ra cosine cao. Thống kê context gốc cũng khác với
context sau subsampling trong quá trình train.

In [6]:
baseline = summaries['sg_d100_w5']
for word, neighbors in baseline['neighbors'].items():
    print(word, 'frequency=', counts[word], 'top5=', neighbors)
evidence = collect_evidence(records, models['sg_d100_w5'], counts, recorder)
for item in evidence['pairs']:
    print(item)
print('Example sentences for doctor and physician:')
for word in ['doctor','physician']:
    print(word, evidence['examples'][word])

doctor frequency= 204 top5= [('physician', 0.7589634656906128), ('anesthesia', 0.7328252792358398), ('dentist', 0.7133238911628723), ('surgeon', 0.6917833685874939), ('ophthalmologist', 0.6897867321968079)]
hospital frequency= 234 top5= [('cardiologist', 0.6619685292243958), ('chongqing', 0.6414457559585571), ('rehabilitation', 0.6349308490753174), ('navicent', 0.6332074403762817), ('pediatrics', 0.632224440574646)]
patient frequency= 232 top5= [('anesthesia', 0.6639977693557739), ('childbirth', 0.6280533075332642), ('doctor', 0.6229605078697205), ('hospice', 0.6159873008728027), ('checkup', 0.6072453856468201)]
disease frequency= 233 top5= [('chronic', 0.8041166663169861), ('diseases', 0.7817336320877075), ('degenerative', 0.7749053835868835), ('kidney', 0.7514569163322449), ('rheumatoid', 0.7501479387283325)]
computer frequency= 341 top5= [('device', 0.6632190942764282), ('encrypt', 0.6348161101341248), ('uploading', 0.6313602924346924), ('settings', 0.6288225650787354), ('desktop', 

## 5. Thay đổi context window

Em giữ dimension=100 và các thông số còn lại, chỉ đổi window 2, 5, 10.
Em so sánh doctor-physician, doctor-hospital và cat-dog.

Doctor-physician tăng từ 0,7143 lên 0,7590, rồi giảm còn 0,6365. Cat-dog
lại giảm từ 0,6668 xuống 0,6130 rồi 0,5694. Vì vậy, em thấy tăng window
không làm tất cả similarity cùng tăng.

Window rộng thu thêm thông tin về chủ đề nhưng cũng có thể lấy vào
context xa và ít đặc trưng. Em cần xem nhiệm vụ sử dụng trước khi chọn
window phù hợp.

In [7]:
for window in [2,5,10]:
    name = f'sg_d100_w{window}'
    model = models[name]
    print(name)
    for a,b in [('doctor','physician'),('doctor','hospital'),('cat','dog')]:
        score = float(model.wv.similarity(a,b)) if a in model.wv and b in model.wv else None
        recorder.add('window_comparison', name, a,b,'cosine',score)
        print(f'  {a}-{b}: {score}')

sg_d100_w2
  doctor-physician: 0.7143160700798035
  doctor-hospital: 0.5244470834732056
  cat-dog: 0.6668489575386047
sg_d100_w5
  doctor-physician: 0.758963406085968
  doctor-hospital: 0.562873899936676
  cat-dog: 0.6129826903343201
sg_d100_w10
  doctor-physician: 0.6365437507629395
  doctor-hospital: 0.5202397108078003
  cat-dog: 0.5694354772567749


## 6. Thay đổi số chiều embedding

Em giữ window=5 và thử 50, 100, 300 chiều. Các tiêu chí gồm thời gian
train, kích thước model, similarity, analogy và tìm kiếm văn bản.

Model 300 chiều lớn hơn model 100 chiều, nhưng accuracy Top-1 analogy
là 12,5%, thấp hơn mức 25% của model 100 chiều. Top-5 lại tăng từ 37,5%
lên 50%. Em thấy cần xem nhiều metric, thay vì chỉ chọn model nhiều chiều.

Bộ search nhỏ cho MRR và Recall@3 bằng 1 ở cả ba cấu hình, nên chưa
phân biệt được chất lượng. Thời gian mới nhất của từng model được ghi
trong bảng tổng hợp ở README.

In [8]:
for dimension in [50,100,300]:
    name = f'sg_d{dimension}_w5'
    s = summaries[name]
    print(name, {k: s[k] for k in ['vocabulary','training_seconds','model_bytes','vector_bytes','spearman_intuition']})
    print('Similarity:', s['pairs'])
    print('Analogy:', s['analogy_metrics'])
    print('Search:', s['search']['aggregates'])

sg_d50_w5 {'vocabulary': 29367, 'training_seconds': 49.141417500097305, 'model_bytes': 12640205, 'vector_bytes': 5873400, 'spearman_intuition': 0.6155870112510925}
Similarity: [{'a': 'doctor', 'b': 'physician', 'score': 0.8560739159584045, 'intuition': 9.5}, {'a': 'car', 'b': 'automobile', 'score': 0.6811845898628235, 'intuition': 9.5}, {'a': 'king', 'b': 'queen', 'score': 0.5468789339065552, 'intuition': 7.5}, {'a': 'cat', 'b': 'dog', 'score': 0.77642822265625, 'intuition': 7.0}, {'a': 'computer', 'b': 'banana', 'score': -0.0006002369336783886, 'intuition': 0.0}]
Analogy: {'covered': 8, 'total': 8, 'accuracy_at_1': 0.125, 'accuracy_at_5': 0.25}
Search: {'tfidf': {'mrr': 0.8333333333333334, 'recall_at_3': 0.6388888888888888}, 'embedding': {'mrr': 1.0, 'recall_at_3': 1.0}}
sg_d100_w5 {'vocabulary': 29367, 'training_seconds': 61.39738410001155, 'model_bytes': 24387008, 'vector_bytes': 11746800, 'spearman_intuition': 0.6155870112510925}
Similarity: [{'a': 'doctor', 'b': 'physician', 'scor

## 7. Đánh giá similarity và analogy

Em xếp hạng 5 cặp từ và so với mức trực giác đã đặt ban đầu:
`[9.5, 9.5, 7.5, 7, 0]`. Spearman của baseline là 0,6156. Doctor-physician
đứng cao và computer-banana đứng thấp là phù hợp; cat-dog cao hơn
car-automobile là một trường hợp em cần xem lại tần suất từ.

Với 8 câu analogy, các từ đều nằm trong vocabulary. Baseline đúng
Top-1 ở 2/8 câu và có đáp án trong Top-5 ở 3/8 câu. Phép king-man+woman
trả về cal ở Top-1, không phải queen.

Lệnh `most_similar` dùng các vector đầu vào đã chuẩn hóa. Đây là điểm
khác với phép cộng trừ trực tiếp các vector giả định trong bài tính tay.
Em xem vector arithmetic là một pattern được học từ dữ liệu, chưa phải
bằng chứng model hiểu quan hệ như người.

In [9]:
for rank, p in enumerate(sorted(baseline['pairs'], key=lambda p: -(p['score'] if p['score'] is not None else -2)),1):
    print(rank,p)
print('Spearman:', baseline['spearman_intuition'])
for item in baseline['analogies']:
    print(item)
print('Aggregates:', baseline['analogy_metrics'])
print('king - man + woman:', models['sg_d100_w5'].wv.most_similar(positive=['king','woman'],negative=['man'],topn=5))

1 {'a': 'doctor', 'b': 'physician', 'score': 0.758963406085968, 'intuition': 9.5}
2 {'a': 'cat', 'b': 'dog', 'score': 0.6129826903343201, 'intuition': 7.0}
3 {'a': 'car', 'b': 'automobile', 'score': 0.5850943326950073, 'intuition': 9.5}
4 {'a': 'king', 'b': 'queen', 'score': 0.47528690099716187, 'intuition': 7.5}
5 {'a': 'computer', 'b': 'banana', 'score': 0.01571662351489067, 'intuition': 0.0}
Spearman: 0.6155870112510925
{'a': 'king', 'b': 'man', 'c': 'woman', 'expected': 'queen', 'covered': True, 'top5': [('cal', 0.47959408164024353), ('slumber', 0.4465290904045105), ('gel', 0.4237085282802582), ('robes', 0.42275306582450867), ('assyrian', 0.42099907994270325)], 'hit1': False, 'hit5': False}
{'a': 'paris', 'b': 'france', 'c': 'italy', 'expected': 'rome', 'covered': True, 'top5': [('vilnius', 0.6256234049797058), ('rooftop', 0.6164050102233887), ('loughrea', 0.6042425632476807), ('idyllic', 0.6034312844276428), ('dpaa', 0.601619303226471)], 'hit1': False, 'hit5': False}
{'a': 'berlin

## 8. Ứng dụng semantic search

Em dùng 12 document và 6 query ngắn, với các document liên quan được
chỉ định trước cho từng query. Word2Vec được train trên C4; 12 document
này chỉ dùng cho tìm kiếm.

Vector query/document là trung bình word vectors có trọng số IDF. IDF
được tính từ document tìm kiếm, không dùng nhãn relevance. Token OOV bị
bỏ qua; nếu không còn token hợp lệ thì vector bằng 0. Em xếp hạng bằng
cosine và chỉ trả các kết quả có score lớn hơn 0.

Baseline embedding đạt MRR=1 và Recall@3=1; TF-IDF đạt lần lượt 0,8333
và 0,6389. Với medical treatment, embedding tìm được các document có
physician, therapy, clinical và patient dù không chứa đúng từ query.

Tập minh họa còn nhỏ và các chủ đề khá rõ, nên em chưa kết luận kết quả
này đúng với mọi ứng dụng tìm kiếm.

In [10]:
print('Documents:')
for doc in DOCS: print(doc)
print('Queries and relevance labels:', QUERIES)
print('Baseline search metrics:', baseline['search']['aggregates'])
for result in baseline['search']['details']:
    print(result)
print('medical treatment Top-3:')
for result in baseline['search']['details']:
    if result['query']=='medical treatment':
        print(result['method'], result['ranking'][:3])

Documents:
('med1', 'The physician provides therapy for patients at the clinic.')
('med2', 'Clinical care and medicine help patients recover from disease.')
('med3', 'The doctor recommends treatment for the patient in hospital.')
('car1', 'The mechanic fixes an automobile engine in the garage.')
('car2', 'A car needs maintenance and replacement of damaged parts.')
('tech1', 'A programmer writes software and debugs computer applications.')
('tech2', 'The developer creates a program using a programming language.')
('sport1', 'The football team scored a goal and won the game.')
('sport2', 'Soccer players trained with their coach before the match.')
('money1', 'Customers deposit money and manage savings in a bank.')
('river1', 'Trees grow on the river bank beside flowing water.')
('food1', 'The banana is a fruit used in a sweet breakfast recipe.')
Queries and relevance labels: [('medical treatment', ['med1', 'med2', 'med3']), ('vehicle repair', ['car1', 'car2']), ('computer program', ['tec

## 9. Từ nhiều nghĩa: bank

Word2Vec dùng một vector cho bank dù câu nói về ngân hàng hay bờ sông.
Trong corpus, bank xuất hiện 250 lần. Cosine(bank, money)=0,4683, còn
cosine(bank, river)=0,1671. Có 12 câu đồng xuất hiện bank/money và 1 câu
bank/river, gợi ý context tài chính nổi bật hơn trong tập dữ liệu này.

Em vẫn cần phân biệt số câu đồng xuất hiện với nhãn nghĩa thật sự.
Những ví dụ dưới đây giúp xem hai cách dùng của bank. Khi tính trung
bình query “bank money” và “bank river”, vector query có thể khác nhau,
nhưng vector riêng của bank vẫn giữ nguyên.

In [11]:
print('bank Top-10:', models['sg_d100_w5'].wv.most_similar('bank',topn=10))
print('bank corpus frequency:', counts['bank'])
for other in ['money','river']:
    print('bank-'+other, float(models['sg_d100_w5'].wv.similarity('bank',other)))
for context in ['money','river']:
    matching = [r for r in records if 'bank' in r['tokens'] and context in r['tokens']]
    print(context, 'co-mentioned sentences=',len(matching))
    for r in matching[:2]: print(r['doc_id'],r['url'],' '.join(r['tokens'][:100]))
recorder.save()
print('Saved results.csv rows:',len(recorder.rows))

bank Top-10: [('phoebus', 0.6977390646934509), ('credit', 0.6640511751174927), ('taxpayer', 0.6511642932891846), ('apex', 0.6456268429756165), ('procredit', 0.6443805694580078), ('slovakia', 0.6434204578399658), ('franc', 0.6416719555854797), ('hsbc', 0.6394152045249939), ('cheque', 0.6368533372879028), ('securities', 0.6326668858528137)]
bank corpus frequency: 250
bank-money 0.46826937794685364
bank-river 0.16709491610527039
money co-mentioned sentences= 12
315 https://ethercapital.com/ for example without a bank how would people send money to each other
849 https://www.consumer-action.org/helpdesk/articles/libor_variable_rate_loan in the libor scandal of mid accusations were made that libor had been manipulated by the banks that determine the benchmark s rate allegedly since the early s to either make money or to make their bank and the banking system as a whole appear stronger
river co-mentioned sentences= 1
718 https://mountainpassessouthafrica.co.za/find-a-pass/western-cape/item/1

## 10. Nhận xét sau thực nghiệm

Qua bài lab, em thấy similarity phụ thuộc vào context trong corpus.
Cosine cao có thể phản ánh gần nghĩa, cùng chủ đề hoặc một context riêng
như tên bệnh viện. Để giải thích kết quả, em cần xem thêm câu nguồn và
tần suất của từng từ.

Tăng window hoặc dimension không chắc cải thiện mọi metric. Với từ
nhiều nghĩa như bank, một vector cố định còn có thể gộp các cách dùng
khác nhau. Đây là lý do em cần tìm hiểu contextual embeddings ở bước tiếp theo.

Ba kết quả phù hợp và ba kết quả bất ngờ được phân tích trong
`error_analysis.md`. Bảng so sánh biểu diễn và phần reflection nằm
trong `reflection.md`.